# 06 — Train Pix2Pix Reverse / 06 Train Pix2Pix Reverse

**Chapter 23 — File 6 of 6 / 第23章 — 第6个文件（共6个）**

---

## Summary / 总结

This script demonstrates **example of pix2pix gan for map to satellite image-to-image translation (reverse)**.

本脚本演示 **example of pix2pix gan for map to satellite image-to-image translation (reverse)**。

---
## Step 1 — example of pix2pix gan for map to satellite image-to-image translation (reverse)

In [ ]:
from numpy import load
from numpy import zeros
from numpy import ones
from numpy.random import randint
from keras.optimizers import Adam
from keras.initializers import RandomNormal
from keras.models import Model
from keras.models import Input
from keras.layers import Conv2D
from keras.layers import Conv2DTranspose
from keras.layers import LeakyReLU
from keras.layers import Activation
from keras.layers import Concatenate
from keras.layers import Dropout
from keras.layers import BatchNormalization
from keras.layers import LeakyReLU
from matplotlib import pyplot

---
## Step 2 — define the discriminator model

In [ ]:
def define_discriminator(image_shape):

---
## Step 3 — weight initialization

In [ ]:
init = RandomNormal(stddev=0.02)

---
## Step 4 — source image input

In [ ]:
in_src_image = Input(shape=image_shape)

---
## Step 5 — target image input

In [ ]:
in_target_image = Input(shape=image_shape)

---
## Step 6 — concatenate images channel-wise

In [ ]:
merged = Concatenate()([in_src_image, in_target_image])

---
## Step 7 — C64

In [ ]:
d = Conv2D(64, (4,4), strides=(2,2), padding='same', kernel_initializer=init)(merged)
	d = LeakyReLU(alpha=0.2)(d)

---
## Step 8 — C128

In [ ]:
d = Conv2D(128, (4,4), strides=(2,2), padding='same', kernel_initializer=init)(d)
	d = BatchNormalization()(d)
	d = LeakyReLU(alpha=0.2)(d)

---
## Step 9 — C256

In [ ]:
d = Conv2D(256, (4,4), strides=(2,2), padding='same', kernel_initializer=init)(d)
	d = BatchNormalization()(d)
	d = LeakyReLU(alpha=0.2)(d)

---
## Step 10 — C512

In [ ]:
d = Conv2D(512, (4,4), strides=(2,2), padding='same', kernel_initializer=init)(d)
	d = BatchNormalization()(d)
	d = LeakyReLU(alpha=0.2)(d)

---
## Step 11 — second last output layer

In [ ]:
d = Conv2D(512, (4,4), padding='same', kernel_initializer=init)(d)
	d = BatchNormalization()(d)
	d = LeakyReLU(alpha=0.2)(d)

---
## Step 12 — patch output

In [ ]:
d = Conv2D(1, (4,4), padding='same', kernel_initializer=init)(d)
	patch_out = Activation('sigmoid')(d)

---
## Step 13 — define model

In [ ]:
model = Model([in_src_image, in_target_image], patch_out)

---
## Step 14 — compile model

In [ ]:
opt = Adam(lr=0.0002, beta_1=0.5)
	model.compile(loss='binary_crossentropy', optimizer=opt, loss_weights=[0.5])
	return model

---
## Step 15 — define an encoder block

In [ ]:
def define_encoder_block(layer_in, n_filters, batchnorm=True):

---
## Step 16 — weight initialization

In [ ]:
init = RandomNormal(stddev=0.02)

---
## Step 17 — add downsampling layer

In [ ]:
g = Conv2D(n_filters, (4,4), strides=(2,2), padding='same', kernel_initializer=init)(layer_in)

---
## Step 18 — conditionally add batch normalization

In [ ]:
if batchnorm:
		g = BatchNormalization()(g, training=True)

---
## Step 19 — leaky relu activation

In [ ]:
g = LeakyReLU(alpha=0.2)(g)
	return g

---
## Step 20 — define a decoder block

In [ ]:
def decoder_block(layer_in, skip_in, n_filters, dropout=True):

---
## Step 21 — weight initialization

In [ ]:
init = RandomNormal(stddev=0.02)

---
## Step 22 — add upsampling layer

In [ ]:
g = Conv2DTranspose(n_filters, (4,4), strides=(2,2), padding='same', kernel_initializer=init)(layer_in)

---
## Step 23 — add batch normalization

In [ ]:
g = BatchNormalization()(g, training=True)

---
## Step 24 — conditionally add dropout

In [ ]:
if dropout:
		g = Dropout(0.5)(g, training=True)

---
## Step 25 — merge with skip connection

In [ ]:
g = Concatenate()([g, skip_in])

---
## Step 26 — relu activation

In [ ]:
g = Activation('relu')(g)
	return g

---
## Step 27 — define the standalone generator model

In [ ]:
def define_generator(image_shape=(256,256,3)):

---
## Step 28 — weight initialization

In [ ]:
init = RandomNormal(stddev=0.02)

---
## Step 29 — image input

In [ ]:
in_image = Input(shape=image_shape)

---
## Step 30 — encoder model

In [ ]:
e1 = define_encoder_block(in_image, 64, batchnorm=False)
	e2 = define_encoder_block(e1, 128)
	e3 = define_encoder_block(e2, 256)
	e4 = define_encoder_block(e3, 512)
	e5 = define_encoder_block(e4, 512)
	e6 = define_encoder_block(e5, 512)
	e7 = define_encoder_block(e6, 512)

---
## Step 31 — bottleneck, no batch norm and relu

In [ ]:
b = Conv2D(512, (4,4), strides=(2,2), padding='same', kernel_initializer=init)(e7)
	b = Activation('relu')(b)

---
## Step 32 — decoder model

In [ ]:
d1 = decoder_block(b, e7, 512)
	d2 = decoder_block(d1, e6, 512)
	d3 = decoder_block(d2, e5, 512)
	d4 = decoder_block(d3, e4, 512, dropout=False)
	d5 = decoder_block(d4, e3, 256, dropout=False)
	d6 = decoder_block(d5, e2, 128, dropout=False)
	d7 = decoder_block(d6, e1, 64, dropout=False)

---
## Step 33 — output

In [ ]:
g = Conv2DTranspose(3, (4,4), strides=(2,2), padding='same', kernel_initializer=init)(d7)
	out_image = Activation('tanh')(g)

---
## Step 34 — define model

In [ ]:
model = Model(in_image, out_image)
	return model

---
## Step 35 — define the combined generator and discriminator model, for updating the generator

In [ ]:
def define_gan(g_model, d_model, image_shape):

---
## Step 36 — make weights in the discriminator not trainable

In [ ]:
for layer in d_model.layers:
		if not isinstance(layer, BatchNormalization):
			layer.trainable = False

---
## Step 37 — define the source image

In [ ]:
in_src = Input(shape=image_shape)

---
## Step 38 — connect the source image to the generator input

In [ ]:
gen_out = g_model(in_src)

---
## Step 39 — connect the source input and generator output to the discriminator input

In [ ]:
dis_out = d_model([in_src, gen_out])

---
## Step 40 — src image as input, generated image and classification output

In [ ]:
model = Model(in_src, [dis_out, gen_out])

---
## Step 41 — compile model

In [ ]:
opt = Adam(lr=0.0002, beta_1=0.5)
	model.compile(loss=['binary_crossentropy', 'mae'], optimizer=opt, loss_weights=[1,100])
	return model

---
## Step 42 — load and prepare training images

In [ ]:
def load_real_samples(filename):

---
## Step 43 — load the compressed arrays

In [ ]:
data = load(filename)

---
## Step 44 — unpack the arrays

In [ ]:
X1, X2 = data['arr_0'], data['arr_1']

---
## Step 45 — scale from [0,255] to [-1,1]

In [ ]:
X1 = (X1 - 127.5) / 127.5
	X2 = (X2 - 127.5) / 127.5

---
## Step 46 — return in reverse order

In [ ]:
return [X2, X1]

---
## Step 47 — select a batch of random samples, returns images and target

In [ ]:
def generate_real_samples(dataset, n_samples, patch_shape):

---
## Step 48 — unpack dataset

In [ ]:
trainA, trainB = dataset

---
## Step 49 — choose random instances

In [ ]:
ix = randint(0, trainA.shape[0], n_samples)

---
## Step 50 — retrieve selected images

In [ ]:
X1, X2 = trainA[ix], trainB[ix]

---
## Step 51 — generate 'real' class labels (1)

In [ ]:
y = ones((n_samples, patch_shape, patch_shape, 1))
	return [X1, X2], y

---
## Step 52 — generate a batch of images, returns images and targets

In [ ]:
def generate_fake_samples(g_model, samples, patch_shape):

---
## Step 53 — generate fake instance

In [ ]:
X = g_model.predict(samples)

---
## Step 54 — create 'fake' class labels (0)

In [ ]:
y = zeros((len(X), patch_shape, patch_shape, 1))
	return X, y

---
## Step 55 — generate samples and save as a plot and save the model

In [ ]:
def summarize_performance(step, g_model, dataset, n_samples=3):

---
## Step 56 — select a sample of input images

In [ ]:
[X_realA, X_realB], _ = generate_real_samples(dataset, n_samples, 1)

---
## Step 57 — generate a batch of fake samples

In [ ]:
X_fakeB, _ = generate_fake_samples(g_model, X_realA, 1)

---
## Step 58 — scale all pixels from [-1,1] to [0,1]

In [ ]:
X_realA = (X_realA + 1) / 2.0
	X_realB = (X_realB + 1) / 2.0
	X_fakeB = (X_fakeB + 1) / 2.0

---
## Step 59 — plot real source images

In [ ]:
for i in range(n_samples):
		pyplot.subplot(3, n_samples, 1 + i)
		pyplot.axis('off')
		pyplot.imshow(X_realA[i])

---
## Step 60 — plot generated target image

In [ ]:
for i in range(n_samples):
		pyplot.subplot(3, n_samples, 1 + n_samples + i)
		pyplot.axis('off')
		pyplot.imshow(X_fakeB[i])

---
## Step 61 — plot real target image

In [ ]:
for i in range(n_samples):
		pyplot.subplot(3, n_samples, 1 + n_samples*2 + i)
		pyplot.axis('off')
		pyplot.imshow(X_realB[i])

---
## Step 62 — save plot to file

In [ ]:
filename1 = 'plot_%06d.png' % (step+1)
	pyplot.savefig(filename1)
	pyplot.close()

---
## Step 63 — save the generator model

In [ ]:
filename2 = 'model_%06d.h5' % (step+1)
	g_model.save(filename2)
	print('>Saved: %s and %s' % (filename1, filename2))

---
## Step 64 — train pix2pix models

In [ ]:
def train(d_model, g_model, gan_model, dataset, n_epochs=100, n_batch=1):

---
## Step 65 — determine the output square shape of the discriminator

In [ ]:
n_patch = d_model.output_shape[1]

---
## Step 66 — unpack dataset

In [ ]:
trainA, trainB = dataset

---
## Step 67 — calculate the number of batches per training epoch

In [ ]:
bat_per_epo = int(len(trainA) / n_batch)

---
## Step 68 — calculate the number of training iterations

In [ ]:
n_steps = bat_per_epo * n_epochs

---
## Step 69 — manually enumerate epochs

In [ ]:
for i in range(n_steps):

---
## Step 70 — select a batch of real samples

In [ ]:
[X_realA, X_realB], y_real = generate_real_samples(dataset, n_batch, n_patch)

---
## Step 71 — generate a batch of fake samples

In [ ]:
X_fakeB, y_fake = generate_fake_samples(g_model, X_realA, n_patch)

---
## Step 72 — update discriminator for real samples

In [ ]:
d_loss1 = d_model.train_on_batch([X_realA, X_realB], y_real)

---
## Step 73 — update discriminator for generated samples

In [ ]:
d_loss2 = d_model.train_on_batch([X_realA, X_fakeB], y_fake)

---
## Step 74 — update the generator

In [ ]:
g_loss, _, _ = gan_model.train_on_batch(X_realA, [y_real, X_realB])

---
## Step 75 — summarize performance

In [ ]:
print('>%d, d1[%.3f] d2[%.3f] g[%.3f]' % (i+1, d_loss1, d_loss2, g_loss))

---
## Step 76 — summarize model performance

In [ ]:
if (i+1) % (bat_per_epo * 10) == 0:
			summarize_performance(i, g_model, dataset)

---
## Step 77 — load image data

In [ ]:
dataset = load_real_samples('maps_256.npz')
print('Loaded', dataset[0].shape, dataset[1].shape)

---
## Step 78 — define input shape based on the loaded dataset

In [ ]:
image_shape = dataset[0].shape[1:]

---
## Step 79 — define the models

In [ ]:
d_model = define_discriminator(image_shape)
g_model = define_generator(image_shape)

---
## Step 80 — define the composite model

In [ ]:
gan_model = define_gan(g_model, d_model, image_shape)

---
## Step 81 — train model

In [ ]:
train(d_model, g_model, gan_model, dataset)

---
## Learning Notes / 学习笔记

- **概念**: example of pix2pix gan for map to satellite image-to-image translation (reverse) 是机器学习中的常用技术。  
  *example of pix2pix gan for map to satellite image-to-image translation (reverse) is a common technique in machine learning.*

- **ML 应用**: 本示例展示了如何在实践中应用该技术。  
  *This example shows how to apply the technique in practice.*

---
## Complete Code / 完整代码一览

Below is the full code for quick reference. / 以下是完整代码，供快速参考。

In [ ]:
# ===============================
# Train Pix2Pix Reverse / 06 Train Pix2Pix Reverse
# Complete Code / 完整代码
# ===============================

# example of pix2pix gan for map to satellite image-to-image translation (reverse)
from numpy import load
from numpy import zeros
from numpy import ones
from numpy.random import randint
from keras.optimizers import Adam
from keras.initializers import RandomNormal
from keras.models import Model
from keras.models import Input
from keras.layers import Conv2D
from keras.layers import Conv2DTranspose
from keras.layers import LeakyReLU
from keras.layers import Activation
from keras.layers import Concatenate
from keras.layers import Dropout
from keras.layers import BatchNormalization
from keras.layers import LeakyReLU
from matplotlib import pyplot

# define the discriminator model
def define_discriminator(image_shape):
	# weight initialization
	init = RandomNormal(stddev=0.02)
	# source image input
	in_src_image = Input(shape=image_shape)
	# target image input
	in_target_image = Input(shape=image_shape)
	# concatenate images channel-wise
	merged = Concatenate()([in_src_image, in_target_image])
	# C64
	d = Conv2D(64, (4,4), strides=(2,2), padding='same', kernel_initializer=init)(merged)
	d = LeakyReLU(alpha=0.2)(d)
	# C128
	d = Conv2D(128, (4,4), strides=(2,2), padding='same', kernel_initializer=init)(d)
	d = BatchNormalization()(d)
	d = LeakyReLU(alpha=0.2)(d)
	# C256
	d = Conv2D(256, (4,4), strides=(2,2), padding='same', kernel_initializer=init)(d)
	d = BatchNormalization()(d)
	d = LeakyReLU(alpha=0.2)(d)
	# C512
	d = Conv2D(512, (4,4), strides=(2,2), padding='same', kernel_initializer=init)(d)
	d = BatchNormalization()(d)
	d = LeakyReLU(alpha=0.2)(d)
	# second last output layer
	d = Conv2D(512, (4,4), padding='same', kernel_initializer=init)(d)
	d = BatchNormalization()(d)
	d = LeakyReLU(alpha=0.2)(d)
	# patch output
	d = Conv2D(1, (4,4), padding='same', kernel_initializer=init)(d)
	patch_out = Activation('sigmoid')(d)
	# define model
	model = Model([in_src_image, in_target_image], patch_out)
	# compile model
	opt = Adam(lr=0.0002, beta_1=0.5)
	model.compile(loss='binary_crossentropy', optimizer=opt, loss_weights=[0.5])
	return model

# define an encoder block
def define_encoder_block(layer_in, n_filters, batchnorm=True):
	# weight initialization
	init = RandomNormal(stddev=0.02)
	# add downsampling layer
	g = Conv2D(n_filters, (4,4), strides=(2,2), padding='same', kernel_initializer=init)(layer_in)
	# conditionally add batch normalization
	if batchnorm:
		g = BatchNormalization()(g, training=True)
	# leaky relu activation
	g = LeakyReLU(alpha=0.2)(g)
	return g

# define a decoder block
def decoder_block(layer_in, skip_in, n_filters, dropout=True):
	# weight initialization
	init = RandomNormal(stddev=0.02)
	# add upsampling layer
	g = Conv2DTranspose(n_filters, (4,4), strides=(2,2), padding='same', kernel_initializer=init)(layer_in)
	# add batch normalization
	g = BatchNormalization()(g, training=True)
	# conditionally add dropout
	if dropout:
		g = Dropout(0.5)(g, training=True)
	# merge with skip connection
	g = Concatenate()([g, skip_in])
	# relu activation
	g = Activation('relu')(g)
	return g

# define the standalone generator model
def define_generator(image_shape=(256,256,3)):
	# weight initialization
	init = RandomNormal(stddev=0.02)
	# image input
	in_image = Input(shape=image_shape)
	# encoder model
	e1 = define_encoder_block(in_image, 64, batchnorm=False)
	e2 = define_encoder_block(e1, 128)
	e3 = define_encoder_block(e2, 256)
	e4 = define_encoder_block(e3, 512)
	e5 = define_encoder_block(e4, 512)
	e6 = define_encoder_block(e5, 512)
	e7 = define_encoder_block(e6, 512)
	# bottleneck, no batch norm and relu
	b = Conv2D(512, (4,4), strides=(2,2), padding='same', kernel_initializer=init)(e7)
	b = Activation('relu')(b)
	# decoder model
	d1 = decoder_block(b, e7, 512)
	d2 = decoder_block(d1, e6, 512)
	d3 = decoder_block(d2, e5, 512)
	d4 = decoder_block(d3, e4, 512, dropout=False)
	d5 = decoder_block(d4, e3, 256, dropout=False)
	d6 = decoder_block(d5, e2, 128, dropout=False)
	d7 = decoder_block(d6, e1, 64, dropout=False)
	# output
	g = Conv2DTranspose(3, (4,4), strides=(2,2), padding='same', kernel_initializer=init)(d7)
	out_image = Activation('tanh')(g)
	# define model
	model = Model(in_image, out_image)
	return model

# define the combined generator and discriminator model, for updating the generator
def define_gan(g_model, d_model, image_shape):
	# make weights in the discriminator not trainable
	for layer in d_model.layers:
		if not isinstance(layer, BatchNormalization):
			layer.trainable = False
	# define the source image
	in_src = Input(shape=image_shape)
	# connect the source image to the generator input
	gen_out = g_model(in_src)
	# connect the source input and generator output to the discriminator input
	dis_out = d_model([in_src, gen_out])
	# src image as input, generated image and classification output
	model = Model(in_src, [dis_out, gen_out])
	# compile model
	opt = Adam(lr=0.0002, beta_1=0.5)
	model.compile(loss=['binary_crossentropy', 'mae'], optimizer=opt, loss_weights=[1,100])
	return model

# load and prepare training images
def load_real_samples(filename):
	# load the compressed arrays
	data = load(filename)
	# unpack the arrays
	X1, X2 = data['arr_0'], data['arr_1']
	# scale from [0,255] to [-1,1]
	X1 = (X1 - 127.5) / 127.5
	X2 = (X2 - 127.5) / 127.5
	# return in reverse order
	return [X2, X1]

# select a batch of random samples, returns images and target
def generate_real_samples(dataset, n_samples, patch_shape):
	# unpack dataset
	trainA, trainB = dataset
	# choose random instances
	ix = randint(0, trainA.shape[0], n_samples)
	# retrieve selected images
	X1, X2 = trainA[ix], trainB[ix]
	# generate 'real' class labels (1)
	y = ones((n_samples, patch_shape, patch_shape, 1))
	return [X1, X2], y

# generate a batch of images, returns images and targets
def generate_fake_samples(g_model, samples, patch_shape):
	# generate fake instance
	X = g_model.predict(samples)
	# create 'fake' class labels (0)
	y = zeros((len(X), patch_shape, patch_shape, 1))
	return X, y

# generate samples and save as a plot and save the model
def summarize_performance(step, g_model, dataset, n_samples=3):
	# select a sample of input images
	[X_realA, X_realB], _ = generate_real_samples(dataset, n_samples, 1)
	# generate a batch of fake samples
	X_fakeB, _ = generate_fake_samples(g_model, X_realA, 1)
	# scale all pixels from [-1,1] to [0,1]
	X_realA = (X_realA + 1) / 2.0
	X_realB = (X_realB + 1) / 2.0
	X_fakeB = (X_fakeB + 1) / 2.0
	# plot real source images
	for i in range(n_samples):
		pyplot.subplot(3, n_samples, 1 + i)
		pyplot.axis('off')
		pyplot.imshow(X_realA[i])
	# plot generated target image
	for i in range(n_samples):
		pyplot.subplot(3, n_samples, 1 + n_samples + i)
		pyplot.axis('off')
		pyplot.imshow(X_fakeB[i])
	# plot real target image
	for i in range(n_samples):
		pyplot.subplot(3, n_samples, 1 + n_samples*2 + i)
		pyplot.axis('off')
		pyplot.imshow(X_realB[i])
	# save plot to file
	filename1 = 'plot_%06d.png' % (step+1)
	pyplot.savefig(filename1)
	pyplot.close()
	# save the generator model
	filename2 = 'model_%06d.h5' % (step+1)
	g_model.save(filename2)
	print('>Saved: %s and %s' % (filename1, filename2))

# train pix2pix models
def train(d_model, g_model, gan_model, dataset, n_epochs=100, n_batch=1):
	# determine the output square shape of the discriminator
	n_patch = d_model.output_shape[1]
	# unpack dataset
	trainA, trainB = dataset
	# calculate the number of batches per training epoch
	bat_per_epo = int(len(trainA) / n_batch)
	# calculate the number of training iterations
	n_steps = bat_per_epo * n_epochs
	# manually enumerate epochs
	for i in range(n_steps):
		# select a batch of real samples
		[X_realA, X_realB], y_real = generate_real_samples(dataset, n_batch, n_patch)
		# generate a batch of fake samples
		X_fakeB, y_fake = generate_fake_samples(g_model, X_realA, n_patch)
		# update discriminator for real samples
		d_loss1 = d_model.train_on_batch([X_realA, X_realB], y_real)
		# update discriminator for generated samples
		d_loss2 = d_model.train_on_batch([X_realA, X_fakeB], y_fake)
		# update the generator
		g_loss, _, _ = gan_model.train_on_batch(X_realA, [y_real, X_realB])
		# summarize performance
		print('>%d, d1[%.3f] d2[%.3f] g[%.3f]' % (i+1, d_loss1, d_loss2, g_loss))
		# summarize model performance
		if (i+1) % (bat_per_epo * 10) == 0:
			summarize_performance(i, g_model, dataset)

# load image data
dataset = load_real_samples('maps_256.npz')
print('Loaded', dataset[0].shape, dataset[1].shape)
# define input shape based on the loaded dataset
image_shape = dataset[0].shape[1:]
# define the models
d_model = define_discriminator(image_shape)
g_model = define_generator(image_shape)
# define the composite model
gan_model = define_gan(g_model, d_model, image_shape)
# train model
train(d_model, g_model, gan_model, dataset)